# Bronze Layer — Raw Ingestion from S3
**What this does:** Reads raw JSON files from S3 and loads them into Delta Lake Bronze tables. No cleaning — data stored exactly as received from the APIs.

In [0]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
# boto3   = Python's official AWS library. Same tool your ingestion scripts use
#           to UPLOAD files to S3. Here we use it to DOWNLOAD them.
# json    = built-in Python tool for converting raw text into Python objects
#           (dicts, lists) and back. JSON is just text — json.loads() makes it usable.
# pandas  = Python's data table library. Think of it as Excel inside Python.
#           We use it as a middleman to build the table before Spark takes over,
#           because Databricks Serverless blocks the usual Spark method (sparkContext).
import boto3
import json
import pandas as pd

# ── FETCH CREDENTIALS FROM DATABRICKS SECRET VAULT ───────────────────────────
# WHY NOT just paste the keys directly here?
# If you hardcode "AKIA123..." in a notebook, anyone who sees the notebook
# (screenshot, shared link, accidental commit) sees your live AWS keys.
# dbutils.secrets.get() fetches the value at runtime from a locked vault —
# even the output prints as [REDACTED], so the real value is never visible.
aws_access_key = dbutils.secrets.get(scope="crypto-pipeline-scope", key="AWS_ACCESS_KEY_ID")
aws_secret_key = dbutils.secrets.get(scope="crypto-pipeline-scope", key="AWS_SECRET_ACCESS_KEY")
s3_bucket      = dbutils.secrets.get(scope="crypto-pipeline-scope", key="S3_BUCKET")

# ── CREATE THE S3 CLIENT ──────────────────────────────────────────────────────
# boto3.client() opens a secure, authenticated connection to AWS S3.
# Think of it as dialing a phone number to the S3 warehouse in Singapore.
# Every file request we make after this goes through this connection.
# region_name must match where your S3 bucket was created (ap-southeast-1 = Singapore).
s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name="ap-southeast-1"
)

# ── QUICK CONNECTION TEST ─────────────────────────────────────────────────────
# Before doing any real work, we ask S3 to list 5 files under prices/.
# If this prints file paths — connection works. If it throws an error — fix it here
# before running the rest of the notebook. Fail fast, fail early.
response = s3.list_objects_v2(Bucket=s3_bucket, Prefix="prices/", MaxKeys=5)
print(f"S3 connected! Files found under prices/: {response.get('KeyCount', 0)}")
for obj in response.get("Contents", []):
    print(" -", obj["Key"])

In [0]:
# ── HELPER FUNCTION 1: READ ALL JSON FILES FROM AN S3 FOLDER ─────────────────
# WHY a function instead of writing the code directly?
# We need to do this exact same process 3 times: prices, fear_greed, news.
# A function lets us write the logic once and call it with different folder names.
# This follows the DRY principle — Don't Repeat Yourself.
def read_json_files_from_s3(prefix):
    """
    Give it a folder name (prefix) like "prices/" and it returns
    a Python list containing the parsed contents of every .json file in that folder.
    """
    all_records = []

    # WHY paginator? S3 returns a maximum of 1000 files per request.
    # If your folder has 5000 files, one request only gives you the first 1000.
    # A paginator automatically keeps requesting the next "page" until
    # all files have been retrieved — like turning pages in a book automatically.
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=s3_bucket, Prefix=prefix)

    for page in pages:
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".json"):
                # s3.get_object() = download the actual file content from S3
                file_obj = s3.get_object(Bucket=s3_bucket, Key=obj["Key"])

                # .read() gets the raw bytes, .decode("utf-8") converts to text string
                raw_text = file_obj["Body"].read().decode("utf-8")

                # json.loads() converts the text string into a Python dict or list
                # At this point "data" is a real Python object we can work with
                data = json.loads(raw_text)

                # Some JSON files contain a LIST of records (e.g. multiple coins)
                # Some contain a SINGLE record object (e.g. one fear/greed reading)
                # extend() adds all items from a list into all_records
                # append() adds one single item into all_records
                if isinstance(data, list):
                    all_records.extend(data)
                else:
                    all_records.append(data)

    return all_records


# ── HELPER FUNCTION 2: CONVERT PYTHON LIST INTO A SPARK DATAFRAME ────────────
# WHY do we need this conversion?
# Spark (the distributed processing engine that powers Databricks) works
# with DataFrames — tables with rows and columns. Our data is currently
# a Python list of dicts. This function bridges that gap.
#
# WHY not just use spark.createDataFrame(records) directly?
# Our JSON has nested fields — arrays or dicts inside other dicts.
# For example, prices files have a "records" field that contains an array
# of 5 coin objects. Spark's createDataFrame() can't guess the type of
# complex nested fields and throws a [CANNOT_INFER_TYPE] error.
# We handle this by flattening/stringifying nested fields first.
def to_spark_df(records):
    flat = []
    for r in records:
        if isinstance(r, dict) and "records" in r:
            # Prices files: each file has {"ingested_at":..., "records": [...coins...]}
            # We extract the inner coin array so each COIN becomes its own row,
            # not each FILE. This is more useful for querying later.
            flat.extend(r["records"])
        elif isinstance(r, dict) and "data" in r:
            # Sentiment files: each file has {"ingested_at":..., "source":..., "data":...}
            # The "data" field from fear_greed API is a list of readings
            data = r["data"]
            if isinstance(data, list):
                flat.extend(data)
            else:
                # For non-list "data" fields, keep the whole row —
                # the nested "data" dict will get stringified below
                flat.append(r)
        else:
            flat.append(r)

    # Stringify any remaining nested types (dicts or lists inside field values)
    # WHY: Spark DataFrame columns must contain simple values — strings, numbers,
    # booleans. If a column value is itself a dict (nested object), Spark can't
    # store it. json.dumps() converts it to a JSON string, which IS a simple value.
    # In the Silver notebook, we'll parse these strings back out when needed.
    cleaned = []
    for row in flat:
        cleaned.append({
            k: (json.dumps(v) if isinstance(v, (dict, list)) else v)
            for k, v in row.items()
        })

    # WHY pandas as the middleman?
    # Databricks Serverless blocks spark.sparkContext (direct JVM access).
    # The normal approach (sparkContext.parallelize()) doesn't work here.
    # The workaround: Python list → pandas DataFrame → Spark DataFrame.
    # spark.createDataFrame() accepts a pandas DataFrame without needing sparkContext.
    return spark.createDataFrame(pd.DataFrame(cleaned))


# ── READ PRICES DATA ──────────────────────────────────────────────────────────
# Calls our helper to download all JSON files from the prices/ folder in S3
prices_records = read_json_files_from_s3("prices/")
print(f"Prices records loaded: {len(prices_records)}")

# Convert to Spark DataFrame using our second helper
df_prices_raw = to_spark_df(prices_records)
print(f"Columns: {df_prices_raw.columns}")
display(df_prices_raw)

In [0]:
# ── READ SENTIMENT DATA ───────────────────────────────────────────────────────
# WHY "fear_greed/" and "news/" and not "sentiment/fear_greed/"?
# Because fetch_sentiment.py uploads with these exact prefixes (lines 203-204):
#   upload_to_s3(fg_path,   s3_prefix="fear_greed")
#   upload_to_s3(news_path, s3_prefix="news")
# So in S3 the folders are at the top level — fear_greed/ and news/ —
# not nested inside a sentiment/ parent folder.
fear_greed_records = read_json_files_from_s3("fear_greed/")
news_records       = read_json_files_from_s3("news/")

print(f"Fear & Greed records loaded: {len(fear_greed_records)}")
print(f"News records loaded: {len(news_records)}")

# Same to_spark_df() helper — works for any source because it handles
# both the "records" key (prices) and the "data" key (sentiment)
df_fear_greed_raw = to_spark_df(fear_greed_records)
df_news_raw       = to_spark_df(news_records)

print(f"Fear & Greed columns: {df_fear_greed_raw.columns}")
print(f"News columns: {df_news_raw.columns}")
display(df_fear_greed_raw)

In [0]:
# ── WRITE ALL 3 DATAFRAMES TO DELTA LAKE BRONZE TABLES ───────────────────────
#
# WHY Delta Lake format instead of plain CSV or JSON files?
# Delta Lake gives you 4 superpowers that plain files don't have:
#   1. ACID transactions — writes either fully succeed or fully fail.
#      No partial writes that corrupt your data if the job crashes halfway.
#   2. Time travel — you can query what the table looked like yesterday
#      with: SELECT * FROM bronze_prices VERSION AS OF 1
#   3. Schema enforcement — if a new file arrives with a wrong column type,
#      Delta rejects it instead of silently corrupting your table.
#   4. SQL queryable — once saved as a table, you can run SQL on it anywhere
#      in Databricks.
#
# WHY mode("overwrite")?
# This makes the notebook idempotent — safe to re-run at any time.
# If the pipeline crashes and you re-run it, the table gets replaced cleanly.
# You never end up with duplicate rows from multiple runs.
# Bronze is meant to reflect the CURRENT state of what's in S3, not accumulate history.
# That accumulation happens in Silver using proper merge/upsert logic.
#
# WHY saveAsTable() instead of .save("/some/path")?
# saveAsTable() registers the table in Databricks' catalog (the central directory
# of all tables). Any notebook in your workspace can then query it with plain SQL
# like: SELECT * FROM bronze_prices
# Without saveAsTable(), the data is saved as files but has no name — you'd need
# to know the exact file path to access it.
df_prices_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_prices")
df_fear_greed_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_fear_greed")
df_news_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_news")

print("Bronze tables written:")
print("  bronze_prices")
print("  bronze_fear_greed")
print("  bronze_news")

In [0]:
# ── VERIFY ───────────────────────────────────────────────────────────────────
# SHOW TABLES lists every table in the current database
# You should see all 3 bronze tables below
display(spark.sql("SHOW TABLES"))